In [31]:
!pip install openai langchain_core langchain_openai

In [32]:
import os
import numpy as np
from numpy import dot
from numpy.linalg import norm
import pandas as pd
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-qwen3-embedding-8b",
    base_url="http://host.docker.internal:12345/v1",
    api_key="lm-studio",
    check_embedding_ctx_length=False,
)

In [33]:
query_result = embeddings.embed_query("저는 배가 고파요")
print(len(query_result))
print(query_result[:5])

4096
[0.01933622546494007, 0.00946054607629776, -0.001324507873505354, -0.04636580869555473, 0.012087306007742882]


In [34]:
data=[
    '주식 시장이 급등했어요',
    '시장 물가가 올랐어요',
    '전통 시장에는 다양한 물품들을 팔아요',
    '부동산 시장이 점점 더 복잡해지고 있어요',
    '저는 빠른 비트를 좋아해요',
    '최근 비트코인 가격이 많이 변동했어요',
]

df = pd.DataFrame(data, columns=['text'])

In [35]:
def get_embedding(text):
    return embeddings.embed_query(text)

df['embedding'] = df.apply(lambda row:get_embedding(row['text']), axis=1)
df

,text,embedding
0,주식 시장이 급등했어요,"[0.004047391004860401, 0.014936055056750774, -..."
1,시장 물가가 올랐어요,"[0.013641850091516972, 0.02231094427406788, -0..."
2,전통 시장에는 다양한 물품들을 팔아요,"[-0.0131897097453475, 0.02173476852476597, -0...."
3,부동산 시장이 점점 더 복잡해지고 있어요,"[0.010680042207241058, 0.01107772532850504, -0..."
4,저는 빠른 비트를 좋아해요,"[0.02971012331545353, 0.0003916164569091052, 0..."
5,최근 비트코인 가격이 많이 변동했어요,"[0.025094613432884216, 0.021152934059500694, 0..."


In [36]:
def cos_sim(A, B):
    return dot(A, B) / (norm(A) * norm(B))

def return_answer_candidate(df, query):
    query_embedding = get_embedding(query)
    df["similarity"] = df.embedding.apply(
        lambda x: cos_sim(np.array(x), np.array(query_embedding))
    )
    return df.sort_values("similarity", ascending=False).head(3)

sim_result = return_answer_candidate(df, '과일 값이 비싸다')
sim_result

,text,embedding,similarity
1,시장 물가가 올랐어요,"[0.013641850091516972, 0.02231094427406788, -0...",0.713859
0,주식 시장이 급등했어요,"[0.004047391004860401, 0.014936055056750774, -...",0.590800
5,최근 비트코인 가격이 많이 변동했어요,"[0.025094613432884216, 0.021152934059500694, 0...",0.571631


In [37]:
def get_detailed_instruct(task: str, query: str) -> str:
    return f'Instruct: {task}\nQuery:{query}'

def get_embedding_instruct(text: str, task: str):   # get_embedding의 인스트럭션 버전
    return embeddings.embed_query(get_detailed_instruct(task, text))

def return_answer_candidate_instruct(df, query, task):
    query_embedding = get_embedding_instruct(query, task)
    sim = df.embedding.apply(
        lambda x: cos_sim(np.array(x), np.array(query_embedding))
    )
    out = df.assign(similarity=sim)
    return out.sort_values("similarity", ascending=False).head(3)

In [38]:
query = '과일 값이 비싸다'

print('=== baseline (no instruction) ===')
print(return_answer_candidate(df, query)[['text', 'similarity']].to_string(), '\n')

tasks = {
    'A_generic' : 'Given a query, retrieve sentences with semantically similar meaning',
    'B_generic' : 'Given a statement about rising prices, retrieve sentences about price increases or cost of living',
    'C_websearch': 'Given a web search query, retrieve relevant passages that answer the query',
}

for name, task in tasks.items():
    print(f'=== {name} ===')
    print(return_answer_candidate_instruct(df, query, task)[['text', 'similarity']].to_string(), '\n')

=== baseline (no instruction) ===
                   text  similarity
1           시장 물가가 올랐어요    0.713859
0          주식 시장이 급등했어요    0.590800
5  최근 비트코인 가격이 많이 변동했어요    0.571631 

=== A_generic ===
                   text  similarity
1           시장 물가가 올랐어요    0.730811
0          주식 시장이 급등했어요    0.600787
5  최근 비트코인 가격이 많이 변동했어요    0.582366 

=== B_generic ===
                   text  similarity
1           시장 물가가 올랐어요    0.588700
5  최근 비트코인 가격이 많이 변동했어요    0.468549
0          주식 시장이 급등했어요    0.440634 

=== C_websearch ===
                   text  similarity
1           시장 물가가 올랐어요    0.401734
5  최근 비트코인 가격이 많이 변동했어요    0.359551
0          주식 시장이 급등했어요    0.296513 



In [39]:
commerce_tasks = {
    'D_place' : 'Given a query, retrieve sentences about marketplaces or places where goods are sold',
    'E_buy' : 'Given a query about buying a product, retrieve sentences about where to buy or sell goods',
}

## TEST1 (순수 인스트럭션 효과) : 쿼리 고정 + 상거래 인스트럭션
# -> 쿼리 자체가 "비싸다(가격)"라는 가격 편향과 싸우는 어려운 테스트
print('### TEST 1 : query="과일 값이 비싸다" ###')
for name, task in commerce_tasks.items():
    print(f'=== {name} ===')
    print(return_answer_candidate_instruct(df, '과일 값이 비싸다', task)[['text','similarity']].to_string(), '\n')

# TEST2 (결정적 테스트): 장소/구매 의도 쿼리 + 상거래 인스트럭션
print('### TEST2: query="과일을 어디서 살 수 있나요" ###')
for name, task in commerce_tasks.items():
    print(f'=== {name} ===')
    print(return_answer_candidate_instruct(df, '과일을 어디서 살 수 있나요', task)[['text', 'similarity']].to_string(), '\n')

### TEST 1 : query="과일 값이 비싸다" ###
=== D_place ===
                   text  similarity
1           시장 물가가 올랐어요    0.526791
2  전통 시장에는 다양한 물품들을 팔아요    0.466951
0          주식 시장이 급등했어요    0.398646 

=== E_buy ===
                   text  similarity
1           시장 물가가 올랐어요    0.445231
2  전통 시장에는 다양한 물품들을 팔아요    0.419701
0          주식 시장이 급등했어요    0.319733 

### TEST2: query="과일을 어디서 살 수 있나요" ###
=== D_place ===
                   text  similarity
2  전통 시장에는 다양한 물품들을 팔아요    0.513112
1           시장 물가가 올랐어요    0.360059
0          주식 시장이 급등했어요    0.284949 

=== E_buy ===
                   text  similarity
2  전통 시장에는 다양한 물품들을 팔아요    0.470695
1           시장 물가가 올랐어요    0.336934
0          주식 시장이 급등했어요    0.259792 

